# 04. Evaluation & Ablation / Comparative Baseline Analysis

This notebook:
1. Shows the already-trained **main LTN model** evaluation metrics.
2. **Trains and evaluates every baseline** (B1–B5) directly in cells.
3. Produces a **side-by-side comparative analysis table** across all models.

## 1. Imports and Configuration

In [1]:
import os, sys, yaml
import torch, numpy as np, pandas as pd

# put src/ on the path
sys.path.insert(0, os.path.abspath('../src'))

from dataset import (load_and_preprocess, split_data, fit_scaler,
                     build_dataloaders, TARGET_COLS)

# Load config
with open('../config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


## 2. Baseline Training & Evaluation Helper Functions
These functions handle model training, validation, metric extraction, and CPU optimizations (like freezing the vision backbones).

In [2]:
from collections import defaultdict
from sklearn.metrics import r2_score
from loss import regression_loss
from predicates import get_tau_dict
from evaluate import calculate_csr_metrics
from baselines import (
    build_xgboost_baseline, apply_xgboost_rules,
    TabularOnlyBaseline, ImageOnlyBaseline,
    FullNeuralNoConservation, FullNeuralWithConservation
)

def _compute_regression_metrics(y_true, y_pred):
    """Compute RMSE, MAE, R2, MAPE in original space (numpy arrays)."""
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(np.mean(np.abs(y_true - y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    mask = y_true > 0
    if mask.sum() > 0:
        mape = float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)
    else:
        mape = float('nan')
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': mape}

def _preds_to_orig_space(preds_dict):
    """Convert log-space predictions to original space."""
    _COL_TO_ORIG = {
        'Dry_Clover_g': '_clover',
        'Dry_Dead_g':   '_dead',
        'Dry_Green_g':  '_green',
        'Dry_Total_g':  '_total',
        'GDM_g':        '_gdm',
    }
    result = {}
    for col, orig_key in _COL_TO_ORIG.items():
        if orig_key in preds_dict:
            result[col] = preds_dict[orig_key]
        elif col in preds_dict:
            result[col] = torch.expm1(preds_dict[col].clamp(min=0))
    return result

def _print_metrics(name, metrics):
    """Pretty-print metrics for a single model."""
    print(f"\nResults for {name}:")
    for col in TARGET_COLS:
        if col in metrics:
            m = metrics[col]
            print(f"  {col}: RMSE={m['RMSE']:.4f}  MAE={m['MAE']:.4f}  R2={m['R2']:.4f}  MAPE={m['MAPE']:.2f}%")
    if 'CSR' in metrics:
        csr = metrics['CSR']
        print("  CSR: ", end="")
        for k, v in csr.items():
            print(f"{k}={v:.4f} ", end="")
        print()

def _train_neural_baseline(model, model_type, train_loader, val_loader, cfg, device, baseline_epochs=None):
    """Train a neural baseline and return evaluation metrics."""
    epochs = baseline_epochs or cfg['training']['epochs']
    delta  = cfg['loss']['huber_delta']

    if device.type == 'cpu':
        print("  [CPU Optimization] Freezing image encoder backbone to accelerate training...")
        img_encoder = None
        if hasattr(model, 'img_encoder'):
            img_encoder = model.img_encoder
        elif hasattr(model, 'core') and hasattr(model.core, 'img_encoder'):
            img_encoder = model.core.img_encoder

        if img_encoder is not None:
            # Freeze EfficientNet features
            for p in img_encoder.eff_backbone.parameters():
                p.requires_grad = False
            # Freeze ViT layers
            for p in img_encoder.vit_encoder_layers.parameters():
                p.requires_grad = False
            if hasattr(img_encoder, 'vit_conv_proj'):
                for p in img_encoder.vit_conv_proj.parameters():
                    p.requires_grad = False

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg['training']['optimizer']['lr_other'],
        weight_decay=cfg['training']['optimizer']['weight_decay_other'],
    )

    best_val_loss = float('inf')
    best_state = None

    for epoch in range(epochs):
        # train
        model.train()
        train_losses = []
        for images, tab_d, targets in train_loader:
            images  = images.to(device)
            targets = targets.to(device)
            tab_d   = {k: v.to(device) for k, v in tab_d.items()}

            optimizer.zero_grad()

            if model_type == 'tabular_only':
                preds, _, _ = model(tab_d)
            elif model_type == 'image_only':
                preds, _, _ = model(images)
            else:
                tau_dict = get_tau_dict(cfg, epoch)
                preds, _, _ = model(images, tab_d, tau_dict)

            loss, _ = regression_loss(preds, targets, TARGET_COLS, delta)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg['training']['grad_clip'])
            optimizer.step()
            train_losses.append(loss.item())

        # validate
        model.eval()
        val_losses = []
        with torch.no_grad():
            for images, tab_d, targets in val_loader:
                images  = images.to(device)
                targets = targets.to(device)
                tab_d   = {k: v.to(device) for k, v in tab_d.items()}

                if model_type == 'tabular_only':
                    preds, _, _ = model(tab_d)
                elif model_type == 'image_only':
                    preds, _, _ = model(images)
                else:
                    tau_dict = get_tau_dict(cfg, epoch)
                    preds, _, _ = model(images, tab_d, tau_dict)

                loss, _ = regression_loss(preds, targets, TARGET_COLS, delta)
                val_losses.append(loss.item())

        mean_val = np.mean(val_losses)
        if mean_val < best_val_loss:
            best_val_loss = mean_val
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 5 == 0 or epoch == 0 or epoch == epochs - 1:
            print(f"  Epoch {epoch+1}/{epochs}  Train={np.mean(train_losses):.4f}  Val={mean_val:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    return _evaluate_neural_model(model, model_type, val_loader, device, cfg)

def _evaluate_neural_model(model, model_type, val_loader, device, cfg):
    """Evaluate any neural model (baseline or main) and return metrics dict."""
    model.eval()
    all_preds_orig   = defaultdict(list)
    all_targets_orig = defaultdict(list)

    with torch.no_grad():
        for images, tab_d, targets in val_loader:
            images  = images.to(device)
            targets = targets.to(device)
            tab_d   = {k: v.to(device) for k, v in tab_d.items()}

            if model_type == 'tabular_only':
                preds, _, _ = model(tab_d)
            elif model_type == 'image_only':
                preds, _, _ = model(images)
            else:
                tau_dict = get_tau_dict(cfg, 0)
                preds, _, _ = model(images, tab_d, tau_dict)

            orig_preds = _preds_to_orig_space(preds)
            for i, col in enumerate(TARGET_COLS):
                t_orig = torch.expm1(targets[:, i].clamp(min=0))
                all_targets_orig[col].append(t_orig.cpu())
                all_preds_orig[col].append(orig_preds[col].cpu())

    metrics = {}
    for col in TARGET_COLS:
        y_true = torch.cat(all_targets_orig[col]).numpy()
        y_pred = torch.cat(all_preds_orig[col]).numpy()
        metrics[col] = _compute_regression_metrics(y_true, y_pred)

    preds_tensor = {k: torch.cat(v) for k, v in all_preds_orig.items()}
    metrics['CSR'] = calculate_csr_metrics(preds_tensor)
    return metrics

def train_and_evaluate_xgboost(train_df, val_df, cfg):
    """Train and evaluate B1: XGBoost (Tabular Only) baseline."""
    print("\n" + "=" * 50)
    print("B1: XGBoost (Tabular Only)")
    print("=" * 50)
    feature_cols = ['Pre_GSHH_NDVI', 'Height_Ave_cm', 'Species_idx', 'State_idx', 'month_idx', 'season_idx']
    X_train = train_df[feature_cols].values
    y_train = train_df[TARGET_COLS].values
    X_val   = val_df[feature_cols].values
    y_val   = val_df[TARGET_COLS].values

    model = build_xgboost_baseline(random_state=cfg['data'].get('random_seed', 42))
    print("  Training XGBoost...")
    model.fit(X_train, y_train)

    y_pred_log = model.predict(X_val)
    y_pred_orig = np.expm1(np.clip(y_pred_log, 0, None))
    y_true_orig = np.expm1(np.clip(y_val, 0, None))

    y_pred_orig = apply_xgboost_rules(y_pred_orig, val_df['State'].values, val_df['Species'].values, TARGET_COLS)

    metrics = {}
    for i, col in enumerate(TARGET_COLS):
        metrics[col] = _compute_regression_metrics(y_true_orig[:, i], y_pred_orig[:, i])

    preds_t = {col: torch.tensor(y_pred_orig[:, i], dtype=torch.float32) for i, col in enumerate(TARGET_COLS)}
    metrics['CSR'] = calculate_csr_metrics(preds_t)
    _print_metrics("B1: XGBoost", metrics)
    return metrics

def evaluate_main_ltn_model(checkpoint_path, val_loader, cfg, device):
    """Load the trained main BiomassLTN model from checkpoint and evaluate it."""
    from model import BiomassLTNModel
    print("\n" + "=" * 50)
    print("Main: BiomassLTN (Full Neuro-Symbolic)")
    print("=" * 50)
    if not os.path.exists(checkpoint_path):
        print("  WARNING: checkpoint not found - skipping main model evaluation")
        return None
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model = BiomassLTNModel(cfg).to(device)
    model.load_state_dict(ckpt['model_state'])
    print("  Loaded checkpoint from", checkpoint_path)
    m = _evaluate_neural_model(model, 'full', val_loader, device, cfg)
    _print_metrics("Main: BiomassLTN", m)
    return m

def build_comparison_table(all_results, output_dir=None):
    """Build a DataFrame comparing all models, optionally save CSV + text report."""
    rows = []
    for model_name, metrics in all_results.items():
        if metrics is None:
            continue
        for col in TARGET_COLS:
            if col in metrics:
                for mname, mval in metrics[col].items():
                    rows.append({'Model': model_name, 'Target': col, 'Metric': mname, 'Value': mval})
        if 'CSR' in metrics:
            for cname, cval in metrics['CSR'].items():
                rows.append({'Model': model_name, 'Target': 'CSR', 'Metric': cname, 'Value': cval})
    df = pd.DataFrame(rows)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        df.to_csv(os.path.join(output_dir, 'comparative_analysis.csv'), index=False)
        
        # Save a text report
        path = os.path.join(output_dir, 'comparative_analysis.txt')
        model_names = [n for n, m in all_results.items() if m is not None]
        with open(path, 'w') as f:
            f.write("=" * 90 + "\n")
            f.write("COMPARATIVE ANALYSIS - Baseline vs Neuro-Symbolic (LTN)\n")
            f.write("=" * 90 + "\n\n")
            for col in TARGET_COLS:
                f.write(f"--- {col} ---\n")
                f.write(f"{'Model':<50} {'RMSE':>8} {'MAE':>8} {'R2':>8} {'MAPE':>8}\n")
                f.write("-" * 90 + "\n")
                for name in model_names:
                    m = all_results[name].get(col)
                    if m:
                        f.write(f"{name:<50} {m['RMSE']:>8.4f} {m['MAE']:>8.4f} {m['R2']:>8.4f} {m['MAPE']:>8.2f}\n")
                f.write("\n")
            f.write("--- Constraint Satisfaction Rates ---\n")
            csr_keys = None
            for name in model_names:
                csr = all_results[name].get('CSR')
                if csr and csr_keys is None:
                    csr_keys = list(csr.keys())
                    hdr = f"{'Model':<50}"
                    for k in csr_keys:
                        hdr += f" {k[:20]:>20}"
                    f.write(hdr + "\n")
                    f.write("-" * (50 + 21 * len(csr_keys)) + "\n")
                if csr:
                    row = f"{name:<50}"
                    for k in csr_keys:
                        row += f" {csr.get(k, float('nan')):>20.4f}"
                    f.write(row + "\n")
        print(f"\nComparative report saved -> {path}")
    return df

## 3. Data Pipeline
Load the same train / val split used by the main training run.

In [3]:
csv_path = os.path.join('..', cfg['data']['train_csv'])
img_root = os.path.join('..', cfg['data']['img_root'])
outputs_dir = os.path.abspath('../outputs')

df_wide, label_encoders = load_and_preprocess(csv_path, img_root, cfg)
train_df, val_df = split_data(df_wide, cfg)
train_df, val_df, scaler = fit_scaler(train_df, val_df, outputs_dir)

train_loader, val_loader = build_dataloaders(train_df, val_df, cfg, epoch=0)
print(f'Train samples: {len(train_df)}, Val samples: {len(val_df)}')

Train samples: 285, Val samples: 72


c:\Users\Faaiz\Desktop\neuro-symbolic ai\src\dataset.py:316: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  weights[high_biomass] = 0.1 + 0.9 * progress


## 4. Main LTN Model - Evaluation (from saved checkpoint)

Load the best checkpoint produced by `main.py` and evaluate on the validation set.

In [4]:
ckpt_path = os.path.join(outputs_dir, 'checkpoints', 'best_model.pt')
main_metrics = evaluate_main_ltn_model(ckpt_path, val_loader, cfg, device)


Main: BiomassLTN (Full Neuro-Symbolic)
  Loaded checkpoint from c:\Users\Faaiz\Desktop\neuro-symbolic ai\outputs\checkpoints\best_model.pt

Results for Main: BiomassLTN:
  Dry_Clover_g: RMSE=3.4761  MAE=2.0906  R2=0.9027  MAPE=52.30%
  Dry_Dead_g: RMSE=11.4135  MAE=6.0988  R2=0.2915  MAPE=70.52%
  Dry_Green_g: RMSE=14.3347  MAE=7.7796  R2=0.7365  MAPE=47.45%
  Dry_Total_g: RMSE=19.3328  MAE=11.6411  R2=0.6028  MAPE=28.18%
  GDM_g: RMSE=14.5865  MAE=8.4839  R2=0.6861  MAPE=26.77%
  CSR: CSR1_GDM_Conservation=1.0000 CSR2_Total_Conservation=1.0000 CSR3_Clover_LTE_GDM=1.0000 CSR4_Total_GTE_Max=1.0000 Symbolic_Drift_g=0.0000 


### Main model evaluation_summary.txt (reference)

In [5]:
summary_file = '../outputs/metrics/evaluation_summary.txt'
if os.path.exists(summary_file):
    with open(summary_file, 'r') as f:
        print(f.read())
else:
    print('Run training first to generate metrics.')

=== Biomass Prediction Evaluation ===

--- Regression Metrics (Original Space) ---

Dry_Clover_g:
  RMSE: 3.4761
  MAE: 2.0906
  R2: 0.9027
  MAPE: 52.2953

Dry_Dead_g:
  RMSE: 11.4135
  MAE: 6.0988
  R2: 0.2915
  MAPE: 70.5210

Dry_Green_g:
  RMSE: 14.3347
  MAE: 7.7796
  R2: 0.7365
  MAPE: 47.4498

Dry_Total_g:
  RMSE: 19.3328
  MAE: 11.6411
  R2: 0.6028
  MAPE: 28.1798

GDM_g:
  RMSE: 14.5865
  MAE: 8.4839
  R2: 0.6861
  MAPE: 26.7705

--- Constraint Satisfaction Rates ---
CSR1_GDM_Conservation: 1.0000
CSR2_Total_Conservation: 1.0000
CSR3_Clover_LTE_GDM: 1.0000
CSR4_Total_GTE_Max: 1.0000
Symbolic_Drift_g: 0.0000



## 5. Baseline Model Training & Evaluation

We train and evaluate five baselines that progressively add capabilities, allowing us to isolate the contribution of each component:

| ID | Model | Image | Tabular | Conservation | LTN |
|:---|:------|:-----:|:-------:|:------------:|:---:|
| B1 | XGBoost + post-hoc rules | No | Yes | No (post-hoc) | No |
| B2 | Neural Tabular Only | No | Yes | No | No |
| B3 | Neural Image Only | Yes | No | No | No |
| B4 | Full Neural (no constraints) | Yes | Yes | No | No |
| B5 | Full Neural + Conservation | Yes | Yes | Yes | No |
| **Main** | **BiomassLTN** | **Yes** | **Yes** | **Yes** | **Yes** |

### B1: XGBoost (Tabular Only + Post-hoc Rules)

In [6]:
b1_metrics = train_and_evaluate_xgboost(train_df, val_df, cfg)


B1: XGBoost (Tabular Only)
  Training XGBoost...

Results for B1: XGBoost:
  Dry_Clover_g: RMSE=6.6685  MAE=3.5143  R2=0.6418  MAPE=74.48%
  Dry_Dead_g: RMSE=11.2412  MAE=6.0182  R2=0.3128  MAPE=73.64%
  Dry_Green_g: RMSE=16.6491  MAE=8.0775  R2=0.6445  MAPE=42.07%
  Dry_Total_g: RMSE=21.8152  MAE=12.4880  R2=0.4942  MAPE=27.80%
  GDM_g: RMSE=13.6570  MAE=8.9634  R2=0.7248  MAPE=27.09%
  CSR: CSR1_GDM_Conservation=0.1528 CSR2_Total_Conservation=0.1528 CSR3_Clover_LTE_GDM=0.9444 CSR4_Total_GTE_Max=0.7500 Symbolic_Drift_g=0.0000 


### B2: Neural Tabular Only

In [7]:
baseline_epochs = cfg['training'].get('baseline_epochs', cfg['training']['epochs'])
print(f"Training B2 for {baseline_epochs} epochs...")
b2_model = TabularOnlyBaseline(cfg).to(device)
b2_metrics = _train_neural_baseline(b2_model, 'tabular_only', train_loader, val_loader, cfg, device, baseline_epochs)

Training B2 for 3 epochs...
  [CPU Optimization] Freezing image encoder backbone to accelerate training...
  Epoch 1/3  Train=9.8874  Val=12.2038
  Epoch 3/3  Train=5.3071  Val=5.5528


### B3: Neural Image Only

In [8]:
print(f"Training B3 for {baseline_epochs} epochs...")
b3_model = ImageOnlyBaseline(cfg).to(device)
b3_metrics = _train_neural_baseline(b3_model, 'image_only', train_loader, val_loader, cfg, device, baseline_epochs)

Training B3 for 3 epochs...
  [CPU Optimization] Freezing image encoder backbone to accelerate training...
  Epoch 1/3  Train=9.0669  Val=10.6947
  Epoch 3/3  Train=1.9709  Val=2.1661


### B4: Full Neural (No Conservation, No LTN)

In [9]:
print(f"Training B4 for {baseline_epochs} epochs...")
b4_model = FullNeuralNoConservation(cfg).to(device)
b4_metrics = _train_neural_baseline(b4_model, 'full', train_loader, val_loader, cfg, device, baseline_epochs)

Training B4 for 3 epochs...
  [CPU Optimization] Freezing image encoder backbone to accelerate training...
  Epoch 1/3  Train=9.4508  Val=7.5769
  Epoch 3/3  Train=2.2116  Val=2.0325


### B5: Full Neural (With Conservation, No LTN)

In [10]:
print(f"Training B5 for {baseline_epochs} epochs...")
b5_model = FullNeuralWithConservation(cfg).to(device)
b5_metrics = _train_neural_baseline(b5_model, 'full', train_loader, val_loader, cfg, device, baseline_epochs)

Training B5 for 3 epochs...
  [CPU Optimization] Freezing image encoder backbone to accelerate training...
  Epoch 1/3  Train=6.5667  Val=3.9866
  Epoch 3/3  Train=2.3188  Val=1.9101


## 6. Comparative Analysis

Collect results from all models and build a side-by-side comparison.

In [11]:
all_results = {
    'B1: XGBoost (Tabular + Rules)':          b1_metrics,
    'B2: Neural Tabular Only':                 b2_metrics,
    'B3: Neural Image Only':                   b3_metrics,
    'B4: Full Neural (No Constraints)':        b4_metrics,
    'B5: Full Neural (Conservation Only)':     b5_metrics,
}
if main_metrics is not None:
    all_results['Main: BiomassLTN (Full Neuro-Symbolic)'] = main_metrics

metrics_dir = os.path.join(outputs_dir, 'metrics')
comparison_df = build_comparison_table(all_results, metrics_dir)

print('\nComparative CSV and TXT saved to', metrics_dir)


Comparative report saved -> c:\Users\Faaiz\Desktop\neuro-symbolic ai\outputs\metrics\comparative_analysis.txt

Comparative CSV and TXT saved to c:\Users\Faaiz\Desktop\neuro-symbolic ai\outputs\metrics


### RMSE Comparison by Target

In [ ]:
rmse_df = comparison_df[comparison_df['Metric'] == 'RMSE'].pivot(
    index='Model', columns='Target', values='Value'
).reindex(columns=TARGET_COLS)
display(rmse_df.style.highlight_min(axis=0, color='#d4edda'))

### R² Comparison by Target

In [ ]:
r2_df = comparison_df[comparison_df['Metric'] == 'R2'].pivot(
    index='Model', columns='Target', values='Value'
).reindex(columns=TARGET_COLS)
display(r2_df.style.highlight_max(axis=0, color='#d4edda'))

### Constraint Satisfaction Rates

In [ ]:
csr_df = comparison_df[comparison_df['Target'] == 'CSR'].pivot(
    index='Model', columns='Metric', values='Value'
)
display(csr_df.style.highlight_max(axis=0, color='#d4edda'))

## 7. Summary

The comparative analysis above demonstrates the value of each component:

- **B1 -> B2**: Switching from XGBoost to a neural tabular encoder.
- **B2 -> B3**: Replacing tabular features with image features.
- **B3 -> B4**: Combining both modalities (multi-modal fusion).
- **B4 -> B5**: Adding the Symbolic Conservation Layer (exact P1/P2).
- **B5 -> Main**: Adding the soft LTN predicate loss (P3–P11).

The CSR table is particularly important: only B5 and the Main LTN model achieve 100% on the structural conservation constraints (CSR1, CSR2) by construction, while unconstrained baselines violate them.

The full comparative results are saved to:
- `outputs/metrics/comparative_analysis.csv`
- `outputs/metrics/comparative_analysis.txt`